<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-01-logica/Aula_08_Notebook_Sistemas_Especialistas_%E2%80%94_Base_de_Conhecie_Rmento_egras_de_Diagn%C3%B3stico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico

Neste notebook implementamos a arquitetura de **Base de Conhecimento Industrial** orientada a objetos para a **Planta de Produção de Biodiesel**. Estruturamos fatos, regras de produção em Cláusulas de Horn, mecanismos de verificação de consistência e exportação de relatórios estruturados.

In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"  # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str       # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int       # 1 a 10 (10 = mais urgente)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)

        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        return self._indice_antecedentes.get(fato_nome, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

# Instanciação e Cadastro das Regras Especialistas da Planta de Biodiesel
bc = BaseConhecimentoSCADA()

# Regra R-01: Runaway Térmico no Reator R-200
bc.adicionar_regra(
    "R-01", ["p1", "t_alta"], "EXOTERMIA_RUNAWAY_REATOR",
    "Exotermia Descontrolada e Sobrepressão no Reator R-200", "CRÍTICA", 10, 0.5,
    "POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201"
)

# Regra R-02: Corte de Alimentação de Metóxido
bc.adicionar_regra(
    "R-02", ["EXOTERMIA_RUNAWAY_REATOR", "v_in_mix"], "TRIP_ALIMENTACAO_METOXIDO",
    "Corte Imediato da Dosagem de Metóxido por Reação Fora de Controle", "CRÍTICA", 10, 0.5,
    "POP-SIS-02: Fechar imediatamente XV-202, desenergizar P-102 e inertizar com N2"
)

# Regra R-03: Fuga de Vapores de Metanol no Setor 100
bc.adicionar_regra(
    "R-03", ["g_alm"], "VAZAMENTO_GAS_METANOL_S100",
    "Detecção de Vapores Inflamáveis/Tóxicos de Metanol no Setor 100", "CRÍTICA", 9, 1.0,
    "POP-SST-03: Cortar XV-102 e XV-202, ligar exaustão e desenergizar bombas P-102"
)

# Regra R-04: Transbordamento no Reator por Óleo Vegetal
bc.adicionar_regra(
    "R-04", ["l_alto", "v_in_oleo"], "TRANSBORDAMENTO_REATOR_R200",
    "Sobrecarga Volumétrica de Óleo Vegetal no Reator de Transesterificação", "CRÍTICA", 9, 1.0,
    "POP-PR-01: Fechar XV-201, desligar bomba de óleo P-101 e reter batelada"
)

# Regra R-05: Inibição de Aquecimento sem Resfriamento de Emergência
bc.adicionar_regra(
    "R-05", ["h1", "not_r1"], "OPERACAO_TERMICA_SEM_SALVAGUARDA",
    "Acionamento de Aquecedor HT-201 sem Circuito de Resfriamento de Emergência Disponível", "CRÍTICA", 9, 1.0,
    "POP-SIS-04: Trip imediato de HT-201 e alarme de manutenção na linha CW-201"
)

# Regra R-06: Risco de Cavitação / Operação a Seco do Agitador
bc.adicionar_regra(
    "R-06", ["l_baixo", "m_reator"], "RISCO_CAVITACAO_AGITADOR_R200",
    "Operação do Agitador sem Carga Hidráulica Mínima no Reator R-200", "ALTA", 8, 2.0,
    "POP-MA-05: Desarmar inversor de AG-201 e bloquear aquecimento HT-201"
)

# Regra R-07: Perda de Biodiesel no Dreno de Glicerina
bc.adicionar_regra(
    "R-07", ["v_glic", "not_i_glic"], "PERDA_BIODIESEL_DRENO_GLICERINA",
    "Drenagem Indevida de Biodiesel Bruto pela Linha de Fundo de Glicerina", "ALTA", 8, 1.5,
    "POP-SEP-02: Fechar válvula proporcional XV-301 e reajustar tempo de decantação"
)

# Regra R-08: Transferência Final sem Fluxo de Lavagem
bc.adicionar_regra(
    "R-08", ["b_final", "not_f_lav"], "IMPUREZA_CATALISADOR_BIODIESEL",
    "Transferência de Biodiesel sem Etapa de Lavagem e Neutralização Concluída", "ALTA", 7, 3.0,
    "POP-PUR-04: Bloquear bomba P-401, fechar XV-401 e restabelecer água desmineralizada"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (BIODIESEL) ===")
print(formatar_tabela(bc.exportar_catalogo()))

assert len(bc.regras) == 8
assert len(bc.obter_regras_por_fato("p1")) >= 1
print("\n[OK] Base de Conhecimento estruturada, indexada e validada com sucesso!")

=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (BIODIESEL) ===
ID   | Prioridade | Severidade | SE (Antecedentes)                     | ENTÃO (Consequente)              | Diagnóstico                                                                           | POP                                                                                
-----+------------+------------+---------------------------------------+----------------------------------+---------------------------------------------------------------------------------------+------------------------------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p1 AND t_alta                         | EXOTERMIA_RUNAWAY_REATOR         | Exotermia Descontrolada e Sobrepressão no Reator R-200                                | POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201    
R-02 | 10         | CRÍTICA    | EXOTERMIA_RUNAWAY_REATOR AND v_in_mix | TRIP_ALIMENT